## File that generates predictions using trained model

(don't change model architecture - have to be the same as trained model - same for data trasformation)

In [4]:
import pandas as pd
import torch
import torch.nn as nn
import pickle
import torch.utils.data as data

# =========================
# MODEL (musi być taki sam)
# =========================


class RegressionModel(nn.Module):

    def __init__(self, num_inputs=12, num_hidden1=512, num_hidden2=256, act_fn=nn.SiLU(), num_outputs=1):
        super().__init__()
        self.linear1 = nn.Linear(num_inputs, num_hidden1)
        self.act_fn = act_fn
        self.linear2 = nn.Linear(num_hidden1, num_hidden2)
        self.linear3 = nn.Linear(num_hidden2, num_outputs)

    def forward(self, x):
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        x = self.act_fn(x)
        x = self.linear3(x)
        return x

Właściwy kod

In [6]:
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("columns.pkl", "rb") as f:
    columns = pickle.load(f)

# =========================
# wczytanie danych -- transformacje musza byc identyczne jak w oryginalnym modelu
# =========================

df = pd.read_csv("evaluation_data.csv")

df = df.drop(columns=["dteday"])
df = df.astype(
    {
        "season": "int8",
        "hr": "int8",
        "holiday": "int8",
        "weekday": "int8",
        "workingday": "int8",
        "weathersit": "int8",
    }
)
# zachowanie kolejności kolumn
df=df[columns]
numerical_cols = ["temp", "atemp", "hum", "windspeed"]
df[numerical_cols] = scaler.transform(df[numerical_cols])

X = torch.tensor(df.values).float()
dataset = torch.utils.data.TensorDataset(X)
loader = data.DataLoader(dataset, batch_size=256, shuffle=False)

# =========================
# wczytanie modelu
# =========================

model = RegressionModel()

model.load_state_dict(torch.load("bike_model.pt"))
model.eval()

# =========================
# predykcja
# =========================

predictions = []

with torch.no_grad():
    for (x,) in loader:
        y_pred = model(x)
        predictions.extend(y_pred.squeeze().numpy())

# =========================
# zapis
# =========================

pd.Series(predictions).to_csv("predictions_better.csv", index=False, header=False)

print("Predictions saved to predictions_better.csv")

Predictions saved to predictions_better.csv
